# Train: ResNet18

When all four checkpoints exist on Drive, open `05_evaluate_and_compare.ipynb` to run every evaluation/Grad-CAM/robustness/statistics section.


## 1. Environment

In [ ]:
import importlib, subprocess, sys

def ensure(pkg, import_name=None, pip_name=None):
    import_name = import_name or pkg
    pip_name = pip_name or pkg
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)

for pkg, imp in [("timm", "timm"), ("opencv-python", "cv2"), ("scikit-learn", "sklearn")]:
    ensure(pkg, imp, pkg)

import torch, torchvision
print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name        :", torch.cuda.get_device_name(0))
    print("CUDA version    :", torch.version.cuda)
else:
    print("No GPU detected - Runtime > Change runtime type > GPU (T4) is recommended.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device     :", DEVICE)

## Keep this tab active

Colab disconnects free-tier runtimes after ~90 minutes of **browser inactivity**, even while a cell is still running. Run the cell below once — it clicks the "Connect" area periodically so idle-detection doesn't fire. It does **not** get around the ~12hr hard session cap, only the inactivity timeout.


In [ ]:
from IPython.display import Javascript
js_code = (
    "function KeepAlive(){"
    "  document.querySelector(\'colab-toolbar-button#connect\')?.click();"
    "}"
    "setInterval(KeepAlive, 60000);"
)
display(Javascript(js_code))
print("Keep-alive timer started (pings every 60s). Leave this tab open in the foreground.")

## 2. Mount Google Drive

Results are written here so they persist across sessions.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Configuration

In [ ]:
import os

DATASET_DIR = "/content/dataset"


DRIVE_RESULTS_DIR = "/content/drive/MyDrive/deepfake_results"

IMG_SIZE       = 224
BATCH_SIZE     = 32
NUM_EPOCHS     = 15
LEARNING_RATE  = 1e-4
WEIGHT_DECAY   = 1e-4
PATIENCE       = 2
RANDOM_SEED    = 42
THRESHOLD      = 0.5
NUM_WORKERS    = 0

DEBUG_MODE = True   # <<<<<<<<<< SET TO False FOR THE REAL RUN >>>>>>>>>>
DEBUG_SUBSET_PER_CLASS = 40
DEBUG_EPOCHS = 2

if DEBUG_MODE:
    EFFECTIVE_EPOCHS = DEBUG_EPOCHS
    SUBSET_PER_CLASS = DEBUG_SUBSET_PER_CLASS
    print("DEBUG_MODE=True -> quick sanity pass only, not a real result.")
else:
    EFFECTIVE_EPOCHS = NUM_EPOCHS
    SUBSET_PER_CLASS = None
    print("DEBUG_MODE=False -> full training run.")

for sub in ["models", "history"]:
    os.makedirs(os.path.join(DRIVE_RESULTS_DIR, sub), exist_ok=True)
print("Checkpoints/history will be saved under:", DRIVE_RESULTS_DIR)

## 4. Reproducibility

In [ ]:
import random
import numpy as np
import torch

def set_seed(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(RANDOM_SEED)
print("Seed set to", RANDOM_SEED)

## 5. Dataset — pick ONE option

In [ ]:

RUN_OPTION_A = False

if RUN_OPTION_A:
    from google.colab import files
    print("Select your dataset .zip")
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    os.makedirs(DATASET_DIR, exist_ok=True)
    import zipfile
    with zipfile.ZipFile(zip_name, "r") as z:
        z.extractall(DATASET_DIR)
    print("Extracted to", DATASET_DIR)

In [ ]:

RUN_OPTION_B = True
DRIVE_DATASET_PATH = "/content/drive/MyDrive/deepfake-and-real-images"

if RUN_OPTION_B:
    DATASET_DIR = DRIVE_DATASET_PATH
    print("DATASET_DIR now points at Drive:", DATASET_DIR)

In [ ]:

RUN_OPTION_C = False

if RUN_OPTION_C:
    from google.colab import files
    print("Select your kaggle.json token")
    uploaded = files.upload()
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    import shutil
    shutil.move("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    subprocess.run(["pip", "install", "-q", "kaggle"], check=True)
    subprocess.run(["kaggle", "datasets", "download",
                     "-d", "manjilkarki/deepfake-and-real-images", "-p", "/content"], check=True)
    import zipfile
    with zipfile.ZipFile("/content/deepfake-and-real-images.zip", "r") as z:
        z.extractall(DATASET_DIR)
    print("Downloaded and extracted to", DATASET_DIR)

## 6. Resolve Train/Validation/Test folders

In [ ]:
def find_split_root(base_dir):
    target_names = {"train", "validation", "valid", "test"}
    for root, dirs, _ in os.walk(base_dir):
        lower_dirs = {d.lower() for d in dirs}
        if len(lower_dirs & target_names) >= 2:
            return root
    return base_dir

assert os.path.exists(DATASET_DIR), f"DATASET_DIR does not exist: {DATASET_DIR}. Run an Option A/B/C cell first."
SPLIT_ROOT = find_split_root(DATASET_DIR)

def resolve_split_dir(root, names):
    for n in names:
        p = os.path.join(root, n)
        if os.path.isdir(p):
            return p
    return None

TRAIN_DIR = resolve_split_dir(SPLIT_ROOT, ["Train", "train"])
VAL_DIR   = resolve_split_dir(SPLIT_ROOT, ["Validation", "validation", "Valid", "valid", "Val", "val"])
TEST_DIR  = resolve_split_dir(SPLIT_ROOT, ["Test", "test"])
for label, d in [("TRAIN_DIR", TRAIN_DIR), ("VAL_DIR", VAL_DIR), ("TEST_DIR", TEST_DIR)]:
    print(label, "->", d)
    if d is None:
        raise RuntimeError(f"Could not locate {label} under {SPLIT_ROOT}.")

def resolve_class_dir(split_dir, names):
    for n in names:
        p = os.path.join(split_dir, n)
        if os.path.isdir(p):
            return p
    return None

def class_dirs(split_dir):
    return (resolve_class_dir(split_dir, ["Real", "real", "REAL"]),
            resolve_class_dir(split_dir, ["Fake", "fake", "FAKE"]))

TRAIN_REAL_DIR, TRAIN_FAKE_DIR = class_dirs(TRAIN_DIR)
VAL_REAL_DIR, VAL_FAKE_DIR = class_dirs(VAL_DIR)
TEST_REAL_DIR, TEST_FAKE_DIR = class_dirs(TEST_DIR)
print("Resolved class folders OK.")

## 7. Data loaders

In [ ]:
import glob, torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class DeepfakeDataset(Dataset):
    def __init__(self, real_dir, fake_dir, transform, subset_per_class=None, seed=RANDOM_SEED):
        exts = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.JPG", "*.JPEG", "*.PNG")
        real_files, fake_files = [], []
        for e in exts:
            real_files += glob.glob(os.path.join(real_dir, e)) if real_dir else []
            fake_files += glob.glob(os.path.join(fake_dir, e)) if fake_dir else []
        rng = random.Random(seed)
        rng.shuffle(real_files); rng.shuffle(fake_files)
        if subset_per_class is not None:
            real_files = real_files[:subset_per_class]
            fake_files = fake_files[:subset_per_class]
        self.samples = [(f, 0) for f in real_files] + [(f, 1) for f in fake_files]
        rng.shuffle(self.samples)
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = self.transform(Image.open(path).convert("RGB"))
        return img, torch.tensor(label, dtype=torch.float32), path

train_ds = DeepfakeDataset(TRAIN_REAL_DIR, TRAIN_FAKE_DIR, train_transform, subset_per_class=SUBSET_PER_CLASS)
val_ds   = DeepfakeDataset(VAL_REAL_DIR, VAL_FAKE_DIR, eval_transform, subset_per_class=SUBSET_PER_CLASS)
print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")

g = torch.Generator(); g.manual_seed(RANDOM_SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, generator=g)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

n_real_train = sum(1 for _, l in train_ds.samples if l == 0)
n_fake_train = sum(1 for _, l in train_ds.samples if l == 1)
imbalance_ratio = max(n_real_train, n_fake_train) / max(1, min(n_real_train, n_fake_train))
if imbalance_ratio > 1.5:
    pos_weight_value = n_real_train / max(1, n_fake_train)
    print(f"Imbalance ratio {imbalance_ratio:.2f} -> pos_weight={pos_weight_value:.3f}")
else:
    pos_weight_value = 1.0
    print(f"Imbalance ratio {imbalance_ratio:.2f} -> unweighted loss")

## 8. Model architecture

In [ ]:
import torch.nn as nn
import torchvision.models as tvm
try:
    import timm
    HAS_TIMM = True
except ImportError:
    HAS_TIMM = False

class BinaryClassifier(nn.Module):
    def __init__(self, backbone, target_layer):
        super().__init__()
        self.backbone = backbone
        self._target_layer = target_layer
    def forward(self, x):
        return self.backbone(x).squeeze(-1)
    def get_target_layer(self):
        return self._target_layer

def build_resnet18():
    m = tvm.resnet18(weights=tvm.ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, 1)
    return BinaryClassifier(m, m.layer4[-1])

def build_resnet50():
    m = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V2)
    m.fc = nn.Linear(m.fc.in_features, 1)
    return BinaryClassifier(m, m.layer4[-1])

def build_efficientnet_b0():
    m = tvm.efficientnet_b0(weights=tvm.EfficientNet_B0_Weights.IMAGENET1K_V1)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, 1)
    return BinaryClassifier(m, m.features[-1])

def build_xception():
    if not HAS_TIMM:
        raise RuntimeError("timm is required for Xception.")
    m = timm.create_model("xception", pretrained=True, num_classes=1)
    target_layer = None
    for _, module in m.named_modules():
        if isinstance(module, nn.Conv2d):
            target_layer = module
    return BinaryClassifier(m, target_layer)

MODEL_BUILDERS = {
    "resnet18": build_resnet18, "resnet50": build_resnet50,
    "efficientnet_b0": build_efficientnet_b0, "xception": build_xception,
}

## 9. Training function

In [ ]:
import time, json
from tqdm.auto import tqdm

# Mixed precision roughly halves epoch time on a T4 with negligible accuracy impact.
USE_AMP = torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, correct, n = 0.0, 0, 0
    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        pbar = tqdm(loader, leave=False)
        for imgs, labels, _ in pbar:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if is_train:
                optimizer.zero_grad()
            with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                logits = model(imgs)
                loss = criterion(logits, labels)
            if is_train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * imgs.size(0)
            preds = (torch.sigmoid(logits.float()) >= THRESHOLD).float()
            correct += (preds == labels).sum().item()
            n += imgs.size(0)
            pbar.set_postfix(loss=loss.item())
    return total_loss / max(1, n), correct / max(1, n)

def train_model(model_name, epochs):
    """Resumable: if a checkpoint + history already exist on Drive for this model
    (e.g. from a session that got disconnected), training picks up from there
    instead of starting over from epoch 1."""
    print(f"\n{'='*70}\nTraining {model_name} (epochs={epochs}, debug={DEBUG_MODE})\n{'='*70}")
    set_seed(RANDOM_SEED)
    model = MODEL_BUILDERS[model_name]().to(DEVICE)
    pos_weight = torch.tensor([pos_weight_value], device=DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    ckpt_path = os.path.join(DRIVE_RESULTS_DIR, "models", f"{model_name}_best.pth")
    hist_path = os.path.join(DRIVE_RESULTS_DIR, "history", f"{model_name}_history.json")

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    start_epoch = 1
    best_val_loss = float("inf")
    epochs_no_improve = 0

    if os.path.exists(ckpt_path) and os.path.exists(hist_path):
        with open(hist_path) as f:
            history = json.load(f)
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        start_epoch = len(history["train_loss"]) + 1
        if history["val_loss"]:
            best_val_loss = min(history["val_loss"])
        print(f"Found existing checkpoint/history -> resuming from epoch {start_epoch} "
              f"(best_val_loss so far = {best_val_loss:.4f}).")
        if start_epoch > epochs:
            print(f"Already trained {start_epoch - 1} epochs (target was {epochs}) - nothing to do.")
            return model, history

    for epoch in range(start_epoch, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer=None)
        history["train_loss"].append(tr_loss); history["val_loss"].append(val_loss)
        history["train_acc"].append(tr_acc);  history["val_acc"].append(val_acc)
        print(f"Epoch {epoch}/{epochs} [{time.time()-t0:.1f}s]  "
              f"train_loss={tr_loss:.4f} train_acc={tr_acc:.4f}  "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

        # Save checkpoint + history EVERY epoch (not just improving ones) so a
        # disconnect never loses more than one epoch of progress.
        with open(hist_path, "w") as f:
            json.dump(history, f)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), ckpt_path)
            print(f"  -> new best (val_loss={val_loss:.4f}), saved to {ckpt_path}")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f"  Early stopping at epoch {epoch}.")
                break

    print("History saved to", hist_path)
    if os.path.exists(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    return model, history

## 10. Train ResNet18

In [ ]:
model, history = train_model("resnet18", epochs=EFFECTIVE_EPOCHS)
print("\nDone (or paused - safe to re-run this cell any time to continue).")
print("Checkpoint + history saved to Drive:")
print(" -", os.path.join(DRIVE_RESULTS_DIR, "models", "resnet18_best.pth"))
print(" -", os.path.join(DRIVE_RESULTS_DIR, "history", "resnet18_history.json"))
print("\nIf your runtime disconnects, just reconnect and re-run this notebook top to bottom -")
print("it will pick up training from the last completed epoch, not from scratch.")